Bronze

In [0]:
from pyspark.sql.functions import col, current_timestamp

catalog_name = "databricks_learning_premium"
schema_name = "default"

base_path = "abfss://raw@stdatalakeexposure001.dfs.core.windows.net"

raw_events_path = f"{base_path}/exposure/events/*/*.csv"

In [0]:
bronze_events_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(raw_events_path)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

bronze_events_df.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{schema_name}.bronze_exposure_events_raw"
)

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.bronze_exposure_events_raw
LIMIT 20;

In [0]:
import numpy as np
import pandas as pd
events_pd = spark.table(
    "databricks_learning_premium.default.bronze_exposure_events_raw"
).toPandas()

events_pd

Silver

In [0]:
events_pd.columns = [
    column.strip().lower()
    for column in events_pd.columns
]

In [0]:
numeric_columns = [
    "latitude",
    "longitude",
    "hazard_score",
    "insured_value_usd",
    "quoted_premium_usd",
    "expected_loss_usd",
    "bound_flag",
]

for column in numeric_columns:
    events_pd[column] = pd.to_numeric(
        events_pd[column],
        errors="coerce",
    )

In [0]:
events_pd["event_date"] = pd.to_datetime(
    events_pd["event_date"],
    errors="coerce",
)

events_pd["generated_timestamp_utc"] = pd.to_datetime(
    events_pd["generated_timestamp_utc"],
    errors="coerce",
)

In [0]:
text_columns = [
    "quote_id",
    "policy_id",
    "account_name",
    "country",
    "iso_code",
    "region",
    "city",
    "asset_type",
    "peril",
    "pricing_variant",
]

for column in text_columns:
    events_pd[column] = (
        events_pd[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

In [0]:
events_pd = events_pd.dropna(
    subset=[
        "quote_id",
        "event_date",
        "country",
        "city",
        "latitude",
        "longitude",
        "peril",
        "hazard_score",
        "insured_value_usd",
    ]
)

In [0]:
events_pd = events_pd.sort_values(
    by=["generated_timestamp_utc"],
    ascending=True,
)

events_pd = events_pd.drop_duplicates(
    subset=["quote_id"],
    keep="last",
)

In [0]:
events_pd["hazard_level"] = pd.cut(
    events_pd["hazard_score"],
    bins=[0, 1, 2, 3, 4, 5],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High",
    ],
    include_lowest=True,
)

In [0]:
events_pd["risk_category"] = pd.cut(
    events_pd["hazard_score"],
    bins=[0, 2, 3, 5],
    labels=[
        "Low",
        "Medium",
        "High",
    ],
    include_lowest=True,
)

In [0]:
events_pd["risk_weight"] = (
    events_pd["hazard_score"] / 5
)

events_pd["risk_weighted_value_usd"] = (
    events_pd["insured_value_usd"]
    * events_pd["risk_weight"]
)

In [0]:
events_pd["premium_rate"] = (
    events_pd["quoted_premium_usd"]
    / events_pd["insured_value_usd"]
)

events_pd["expected_loss_ratio"] = (
    events_pd["expected_loss_usd"]
    / events_pd["quoted_premium_usd"]
)

In [0]:
events_pd["bound_status"] = np.where(
    events_pd["bound_flag"] == 1,
    "Bound",
    "Not Bound",
)

In [0]:
events_pd["event_month"] = (
    events_pd["event_date"]
    .dt.to_period("M")
    .astype(str)
)

In [0]:
silver_events_spark_df = spark.createDataFrame(events_pd)

silver_events_spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "databricks_learning_premium.default.silver_exposure_events_cleaned"
)

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_exposure_events_cleaned
LIMIT 20;

Forecasting

In [0]:
daily_summary_pd = (
    events_pd
    .groupby("event_date")
    .agg(
        quote_count=("quote_id", "nunique"),
        bound_count=("bound_flag", "sum"),
        total_insured_value_usd=("insured_value_usd", "sum"),
        total_risk_weighted_value_usd=("risk_weighted_value_usd", "sum"),
        avg_hazard_score=("hazard_score", "mean"),
        avg_expected_loss_ratio=("expected_loss_ratio", "mean"),
    )
    .reset_index()
)

daily_summary_pd["bind_rate"] = (
    daily_summary_pd["bound_count"]
    / daily_summary_pd["quote_count"]
)

In [0]:
daily_summary_pd = daily_summary_pd.sort_values("event_date")

daily_summary_pd["rolling_7_day_risk_weighted_value_usd"] = (
    daily_summary_pd["total_risk_weighted_value_usd"]
    .rolling(window=7, min_periods=1)
    .mean()
)

In [0]:
# Make sure the number columns are proper numeric types
daily_summary_pd["quote_count"] = daily_summary_pd["quote_count"].astype("int64")
daily_summary_pd["bound_count"] = daily_summary_pd["bound_count"].astype("int64")

daily_summary_pd["total_insured_value_usd"] = daily_summary_pd["total_insured_value_usd"].astype("float64")
daily_summary_pd["total_risk_weighted_value_usd"] = daily_summary_pd["total_risk_weighted_value_usd"].astype("float64")
daily_summary_pd["avg_hazard_score"] = daily_summary_pd["avg_hazard_score"].astype("float64")
daily_summary_pd["avg_expected_loss_ratio"] = daily_summary_pd["avg_expected_loss_ratio"].astype("float64")
daily_summary_pd["bind_rate"] = daily_summary_pd["bind_rate"].astype("float64")
daily_summary_pd["rolling_7_day_risk_weighted_value_usd"] = daily_summary_pd["rolling_7_day_risk_weighted_value_usd"].astype("float64")

# Convert Pandas back to Spark
daily_summary_spark_df = spark.createDataFrame(daily_summary_pd)

# Overwrite the table AND overwrite the schema
daily_summary_spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("databricks_learning_premium.default.silver_daily_exposure_summary")

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_daily_exposure_summary
ORDER BY event_date;

A/B Testing

In [0]:
ab_summary_pd = (
    events_pd
    .groupby("pricing_variant")
    .agg(
        quote_count=("quote_id", "nunique"),
        bound_count=("bound_flag", "sum"),
        total_quoted_premium_usd=("quoted_premium_usd", "sum"),
        avg_quoted_premium_usd=("quoted_premium_usd", "mean"),
        avg_premium_rate=("premium_rate", "mean"),
        avg_expected_loss_ratio=("expected_loss_ratio", "mean"),
        total_insured_value_usd=("insured_value_usd", "sum"),
        bound_insured_value_usd=(
            "insured_value_usd",
            lambda values: values[events_pd.loc[values.index, "bound_flag"] == 1].sum(),
        ),
    )
    .reset_index()
)

ab_summary_pd["bind_rate"] = (
    ab_summary_pd["bound_count"]
    / ab_summary_pd["quote_count"]
)

In [0]:
ab_summary_spark_df = spark.createDataFrame(ab_summary_pd)

ab_summary_spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "databricks_learning_premium.default.silver_ab_pricing_summary"
)

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_ab_pricing_summary;

In [0]:
ab_summary_pd = (
    events_pd
    .groupby("pricing_variant")
    .agg(
        quote_count=("quote_id", "nunique"),
        bound_count=("bound_flag", "sum"),
        total_quoted_premium_usd=("quoted_premium_usd", "sum"),
        avg_quoted_premium_usd=("quoted_premium_usd", "mean"),
        avg_premium_rate=("premium_rate", "mean"),
        avg_expected_loss_ratio=("expected_loss_ratio", "mean"),
        total_insured_value_usd=("insured_value_usd", "sum"),
        bound_insured_value_usd=(
            "insured_value_usd",
            lambda values: values[events_pd.loc[values.index, "bound_flag"] == 1].sum(),
        ),
    )
    .reset_index()
)

ab_summary_pd["bind_rate"] = (
    ab_summary_pd["bound_count"]
    / ab_summary_pd["quote_count"]
)

In [0]:
ab_summary_spark_df = spark.createDataFrame(ab_summary_pd)

ab_summary_spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "databricks_learning_premium.default.silver_ab_pricing_summary"
)

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_ab_pricing_summary;

In [0]:
# =========================================
# Create smoother forecast table
# =========================================

forecast_input_pd = daily_summary_pd.copy()

forecast_input_pd = forecast_input_pd.sort_values("event_date")
forecast_input_pd["day_number"] = range(len(forecast_input_pd))

# Create 7-day rolling average from actual daily risk-weighted value
forecast_input_pd["rolling_7_day_risk_weighted_value_usd"] = (
    forecast_input_pd["total_risk_weighted_value_usd"]
    .rolling(window=7, min_periods=1)
    .mean()
)

# Use only the most recent 30 days to estimate the trend
recent_history_pd = forecast_input_pd.tail(30).copy()

if len(recent_history_pd) >= 2:
    slope, intercept = np.polyfit(
        recent_history_pd["day_number"],
        recent_history_pd["rolling_7_day_risk_weighted_value_usd"],
        1,
    )

    average_recent_value = recent_history_pd[
        "rolling_7_day_risk_weighted_value_usd"
    ].mean()

    # Stop the forecast from climbing or dropping too aggressively
    max_daily_slope = average_recent_value * 0.01

    slope = np.clip(
        slope,
        -max_daily_slope,
        max_daily_slope,
    )
else:
    slope = 0
    intercept = forecast_input_pd[
        "rolling_7_day_risk_weighted_value_usd"
    ].iloc[-1]

# Difference between noisy actual value and smoother rolling average
forecast_input_pd["residual"] = (
    forecast_input_pd["total_risk_weighted_value_usd"]
    - forecast_input_pd["rolling_7_day_risk_weighted_value_usd"]
)

residual_std = forecast_input_pd["residual"].std()

if pd.isna(residual_std):
    residual_std = 0

last_date = forecast_input_pd["event_date"].max()
last_day_number = forecast_input_pd["day_number"].max()

future_rows = []

# Forecast 90 days forward
for day_offset in range(1, 91):
    future_date = last_date + pd.Timedelta(days=day_offset)
    future_day_number = last_day_number + day_offset

    forecast_value = intercept + slope * future_day_number
    forecast_value = max(forecast_value, 0)

    lower_bound = max(forecast_value - residual_std, 0)
    upper_bound = forecast_value + residual_std

    future_rows.append(
        {
            "event_date": future_date,
            "record_type": "Forecast",
            "actual_risk_weighted_value_usd": None,
            "rolling_7_day_risk_weighted_value_usd": None,
            "forecast_risk_weighted_value_usd": forecast_value,
            "forecast_lower_bound_usd": lower_bound,
            "forecast_upper_bound_usd": upper_bound,
        }
    )

# Actual historical rows
actual_rows = forecast_input_pd[
    [
        "event_date",
        "total_risk_weighted_value_usd",
        "rolling_7_day_risk_weighted_value_usd",
    ]
].copy()

actual_rows["record_type"] = "Actual"

actual_rows["actual_risk_weighted_value_usd"] = actual_rows[
    "total_risk_weighted_value_usd"
]

actual_rows["forecast_risk_weighted_value_usd"] = None
actual_rows["forecast_lower_bound_usd"] = None
actual_rows["forecast_upper_bound_usd"] = None

actual_rows = actual_rows[
    [
        "event_date",
        "record_type",
        "actual_risk_weighted_value_usd",
        "rolling_7_day_risk_weighted_value_usd",
        "forecast_risk_weighted_value_usd",
        "forecast_lower_bound_usd",
        "forecast_upper_bound_usd",
    ]
]

forecast_pd = pd.concat(
    [
        actual_rows,
        pd.DataFrame(future_rows),
    ],
    ignore_index=True,
)

forecast_spark_df = spark.createDataFrame(forecast_pd)

forecast_spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "databricks_learning_premium.default.silver_risk_forecast"
    )

In [0]:
forecast_spark_df = spark.createDataFrame(forecast_pd)

forecast_spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "databricks_learning_premium.default.silver_risk_forecast"
    )

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_risk_forecast
ORDER BY event_date;

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Start from the cleaned Pandas DataFrame
ml_pd = events_pd.copy()

# Keep only rows where the target exists
ml_pd = ml_pd.dropna(subset=["bound_flag"])
ml_pd["bound_flag"] = ml_pd["bound_flag"].astype(int)

categorical_features = [
    "country",
    "region",
    "asset_type",
    "peril",
    "pricing_variant",
    "risk_category",
]

numeric_features = [
    "hazard_score",
    "insured_value_usd",
    "quoted_premium_usd",
    "expected_loss_usd",
    "risk_weighted_value_usd",
    "premium_rate",
    "expected_loss_ratio",
]

feature_columns = categorical_features + numeric_features

target_column = "bound_flag"

# Keep rows with the required feature columns
ml_model_pd = ml_pd[
    [
        "quote_id",
        "policy_id",
        "event_date",
        "country",
        "region",
        "city",
        "asset_type",
        "peril",
        "pricing_variant",
        "insured_value_usd",
        "quoted_premium_usd",
        "expected_loss_usd",
        "risk_weighted_value_usd",
        "premium_rate",
        "expected_loss_ratio",
        "hazard_score",
        "risk_category",
        "bound_flag",
    ]
].copy()

X = ml_model_pd[feature_columns]
y = ml_model_pd[target_column]

# This protects the notebook from failing if the dataset is still too small
if len(ml_model_pd) < 50 or y.nunique() < 2:
    print("Not enough data to train the ML model yet.")
    print("Create a larger seed dataset first, for example BATCH_SIZE=1000.")

    ml_model_pd["predicted_bind_probability"] = None
    ml_model_pd["predicted_bound_flag"] = None
    ml_model_pd["model_name"] = "logistic_regression_bind_probability"
    ml_model_pd["model_status"] = "not_trained_not_enough_data"

    metrics_pd = pd.DataFrame(
        [
            {
                "model_name": "logistic_regression_bind_probability",
                "model_status": "not_trained_not_enough_data",
                "training_row_count": len(ml_model_pd),
                "test_row_count": 0,
                "accuracy": None,
                "roc_auc": None,
            }
        ]
    )

else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(max_iter=1000),
            ),
        ]
    )

    model.fit(X_train, y_train)

    test_predictions = model.predict(X_test)
    test_probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, test_predictions)
    roc_auc = roc_auc_score(y_test, test_probabilities)

    all_probabilities = model.predict_proba(X)[:, 1]
    all_predictions = (all_probabilities >= 0.5).astype(int)

    ml_model_pd["predicted_bind_probability"] = all_probabilities
    ml_model_pd["predicted_bound_flag"] = all_predictions
    ml_model_pd["model_name"] = "logistic_regression_bind_probability"
    ml_model_pd["model_status"] = "trained"

    metrics_pd = pd.DataFrame(
        [
            {
                "model_name": "logistic_regression_bind_probability",
                "model_status": "trained",
                "training_row_count": len(X_train),
                "test_row_count": len(X_test),
                "accuracy": accuracy,
                "roc_auc": roc_auc,
            }
        ]
    )

# Write predictions to Silver
ml_predictions_spark_df = spark.createDataFrame(ml_model_pd)

ml_predictions_spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "databricks_learning_premium.default.silver_bind_probability_predictions"
)

# Write model metrics to Silver
ml_metrics_spark_df = spark.createDataFrame(metrics_pd)

ml_metrics_spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "databricks_learning_premium.default.silver_ml_model_metrics"
)

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_bind_probability_predictions
LIMIT 20;

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.silver_ml_model_metrics;

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_learning_premium.default.gold_exposure_events_engineering AS
SELECT
    quote_id,
    policy_id,
    CAST(event_date AS DATE) AS event_date,
    generated_timestamp_utc,
    account_name,
    country,
    iso_code,
    region,
    city,
    CAST(latitude AS DOUBLE) AS latitude,
    CAST(longitude AS DOUBLE) AS longitude,
    asset_type,
    peril,
    CAST(hazard_score AS DOUBLE) AS hazard_score,
    hazard_level,
    risk_category,
    CAST(insured_value_usd AS DOUBLE) AS insured_value_usd,
    pricing_variant,
    CAST(quoted_premium_usd AS DOUBLE) AS quoted_premium_usd,
    CAST(expected_loss_usd AS DOUBLE) AS expected_loss_usd,
    CAST(bound_flag AS INT) AS bound_flag,
    bound_status,
    CAST(risk_weight AS DOUBLE) AS risk_weight,
    CAST(risk_weighted_value_usd AS DOUBLE) AS risk_weighted_value_usd,
    CAST(premium_rate AS DOUBLE) AS premium_rate,
    CAST(expected_loss_ratio AS DOUBLE) AS expected_loss_ratio,
    event_month
FROM databricks_learning_premium.default.silver_exposure_events_cleaned
WHERE quote_id IS NOT NULL
  AND country IS NOT NULL
  AND city IS NOT NULL
  AND latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND peril IS NOT NULL;

In [0]:
%sql

CREATE OR REPLACE TABLE databricks_learning_premium.default.gold_daily_forecast_engineering AS
SELECT
    CAST(event_date AS DATE) AS event_date,
    record_type,

    CAST(actual_risk_weighted_value_usd AS DOUBLE) AS actual_risk_weighted_value_usd,

    CAST(
        rolling_7_day_risk_weighted_value_usd AS DOUBLE
    ) AS rolling_7_day_risk_weighted_value_usd,

    CAST(
        forecast_risk_weighted_value_usd AS DOUBLE
    ) AS forecast_risk_weighted_value_usd,

    CAST(forecast_lower_bound_usd AS DOUBLE) AS forecast_lower_bound_usd,
    CAST(forecast_upper_bound_usd AS DOUBLE) AS forecast_upper_bound_usd,

    CAST(
        COALESCE(
            actual_risk_weighted_value_usd,
            forecast_risk_weighted_value_usd
        ) AS DOUBLE
    ) AS total_risk_weighted_value_usd

FROM databricks_learning_premium.default.silver_risk_forecast;

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_learning_premium.default.gold_ab_pricing_engineering AS
SELECT
    pricing_variant,
    quote_count,
    bound_count,
    total_quoted_premium_usd,
    avg_quoted_premium_usd,
    avg_premium_rate,
    avg_expected_loss_ratio,
    total_insured_value_usd,
    bound_insured_value_usd,
    bind_rate
FROM databricks_learning_premium.default.silver_ab_pricing_summary;

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_learning_premium.default.gold_bind_probability_predictions_engineering AS
SELECT
    quote_id,
    policy_id,
    CAST(event_date AS DATE) AS event_date,
    country,
    region,
    city,
    asset_type,
    peril,
    pricing_variant,
    CAST(insured_value_usd AS DOUBLE) AS insured_value_usd,
    CAST(quoted_premium_usd AS DOUBLE) AS quoted_premium_usd,
    CAST(expected_loss_usd AS DOUBLE) AS expected_loss_usd,
    CAST(risk_weighted_value_usd AS DOUBLE) AS risk_weighted_value_usd,
    CAST(premium_rate AS DOUBLE) AS premium_rate,
    CAST(expected_loss_ratio AS DOUBLE) AS expected_loss_ratio,
    CAST(hazard_score AS DOUBLE) AS hazard_score,
    risk_category,
    CAST(bound_flag AS INT) AS actual_bound_flag,
    CAST(predicted_bind_probability AS DOUBLE) AS predicted_bind_probability,
    CAST(predicted_bound_flag AS INT) AS predicted_bound_flag,
    model_name,
    model_status
FROM databricks_learning_premium.default.silver_bind_probability_predictions;

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_learning_premium.default.gold_ml_model_metrics_engineering AS
SELECT
    model_name,
    model_status,
    training_row_count,
    test_row_count,
    CAST(accuracy AS DOUBLE) AS accuracy,
    CAST(roc_auc AS DOUBLE) AS roc_auc
FROM databricks_learning_premium.default.silver_ml_model_metrics;

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.gold_exposure_events_engineering
LIMIT 20;

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.gold_daily_forecast_engineering
ORDER BY event_date;

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.gold_ab_pricing_engineering;

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.gold_bind_probability_predictions_engineering
LIMIT 20;

In [0]:
%sql
SELECT *
FROM databricks_learning_premium.default.gold_ml_model_metrics_engineering;